## Setup:

In [ ]:
%%capture
!pip install faiss-gpu google-cloud-bigquery google-cloud-storage \
             gcsfs pyarrow pandas torch --quiet

In [ ]:
!pip install faiss-cpu

In [ ]:
!git clone -b dev https://github.com/sbnikhil/RecoSys.git
!cd RecoSys

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded")

# Use the real filename from the upload
fname = next(iter(uploaded.keys()))
path = os.path.abspath(fname)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = path
print("GOOGLE_APPLICATION_CREDENTIALS =", path)

In [ ]:
import sys
%cd /content/RecoSys
sys.path.insert(0, '/content/RecoSys')
!ls

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Upload the artifacts/500k/ folder contents from your local machine.
# Easiest: zip it locally, upload here, unzip.
# Run this on your LOCAL machine first:
#   cd artifacts && zip -r 500k_artifacts.zip 500k/
# Then upload the zip here:

from google.colab import files
import zipfile

print("Upload 500k.zip:")
uploaded = files.upload()
with zipfile.ZipFile("500k.zip", "r") as z:
    z.extractall("artifacts/")
print("Artifacts extracted:")
!ls artifacts/500k/

In [ ]:
import pickle
import pandas as pd

ARTIFACTS_DIR = "artifacts/500k/"
GCS_TEST_PATH = "gs://recosys-data-bucket/samples/users_sample_500k/test/"

with open(f"{ARTIFACTS_DIR}vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

items_enc   = pd.read_parquet(f"{ARTIFACTS_DIR}items_encoded.parquet")
users_enc   = pd.read_parquet(f"{ARTIFACTS_DIR}users_encoded.parquet")
train_pairs = pd.read_parquet(f"{ARTIFACTS_DIR}train_pairs.parquet")
test_df     = pd.read_parquet(GCS_TEST_PATH)

print(f"items_encoded : {items_enc.shape}")
print(f"users_encoded : {users_enc.shape}")
print(f"train_pairs   : {train_pairs.shape}")
print(f"test_df       : {test_df.shape}")
print(f"n_users       : {len(vocabs['user2idx']):,}")
print(f"n_items       : {len(vocabs['item2idx']):,}")


## GRU4Rec Training:

In [ ]:
# If you need to rebuild the session sequences on Colab first:
!python scripts/sequence/build_session_sequences.py

In [ ]:
# Then kick off overnight training:
!python scripts/sequence/train_gru4rec_session.py
# A100 default: batch_size=256 (~11 GB peak VRAM)
# T4/V100: add --batch-size 64 or --batch-size 128